# CP2/CP3 — Anonimizador LGPD com Qwen3-0.6B + LoRA (treino no Colab)

1. Em **Ambiente de execução → Alterar o tipo de ambiente de execução**, escolha **GPU (T4)**.
2. Preencha `REPO_URL` com o link do repositório do grupo no GitHub (ou deixe vazio para enviar um `.zip` do projeto).
3. Rode as células em ordem. Os resultados são salvos no Google Drive, então nada se perde se a sessão cair: é só rodar de novo, porque os experimentos prontos são pulados.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/cp-llm-lgpd'
!mkdir -p "$DRIVE_DIR"

In [ ]:
REPO_URL = ''  # ex.: 'https://github.com/seu-usuario/cp-llm-lgpd.git'

import os
%cd /content
if REPO_URL:
    if not os.path.exists('/content/projeto'):
        !git clone "$REPO_URL" projeto
else:
    from google.colab import files
    up = files.upload()  # envie o .zip do projeto
    nome = list(up)[0]
    !unzip -oq "$nome" -d projeto
    # se o zip tiver uma pasta raiz, entra nela
    itens = os.listdir('/content/projeto')
    if len(itens) == 1 and os.path.isdir(f'/content/projeto/{itens[0]}'):
        !mv /content/projeto/{itens[0]}/* /content/projeto/ && rmdir /content/projeto/{itens[0]}
%cd /content/projeto
!ls

In [ ]:
!pip install -q -r requirements-colab.txt

Restaura resultados de uma sessão anterior (se houver).

In [ ]:
!mkdir -p outputs results
!cp -rn "$DRIVE_DIR/outputs/." outputs/ 2>/dev/null || true
!cp -rn "$DRIVE_DIR/results/." results/ 2>/dev/null || true

In [ ]:
!python scripts/gpu_check.py

## 1. Dataset sintético

In [ ]:
!python data/gerar_dataset.py
!head -n 3 data/train.jsonl

## 2. Teste rápido (≈2 min) — confere se tudo funciona antes do treino completo

In [ ]:
!python src/train.py --out /tmp/smoke --train-limit 64 --epochs 1 && python src/evaluate.py --nome smoke --adapter /tmp/smoke --limit 16 --out-dir /tmp/smoke_res

## 3. Todos os experimentos (base + 5 configurações de LoRA + temperaturas)

Salva no Drive ao final. Se a sessão cair, rode as células de cima de novo e esta continua de onde parou.

In [ ]:
try:
    !python scripts/experimentos.py
finally:
    !mkdir -p "$DRIVE_DIR/outputs" "$DRIVE_DIR/results"
    !cp -r outputs/. "$DRIVE_DIR/outputs/"
    !cp -r results/. "$DRIVE_DIR/results/"

## 4. Resultados

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/resumo.md', encoding='utf-8').read()))
import os
if os.path.exists('results/curvas_loss.png'):
    display(Image('results/curvas_loss.png'))

## 5. Baixar o melhor adaptador + resultados

Descompacte na raiz do projeto no seu PC (pastas `outputs/melhor` e `results/`) para rodar o frontend localmente.

In [ ]:
!zip -qr resultados_colab.zip outputs/melhor results
!cp resultados_colab.zip "$DRIVE_DIR/"
from google.colab import files
files.download('resultados_colab.zip')

Ao terminar: **Ambiente de execução → Desconectar e excluir ambiente de execução** para não gastar unidades de computação.